# Stage 12 — Guide-level pseudobulk QC (GSE270828)
**No Google Drive account connection required.**

This notebook only aggregates the original raw integer counts by **replicate × individual guide** from the public PRISM processed GSE270828 h5ad. It retains every gene, keeps `single_true` cells, and checks that each guide maps uniquely to its annotated HAR. It computes no P-values or validated HAR–gene targets.

**Important:** Download is ~4 GB. Choose High-RAM if available, and ensure ~6 GB free Colab disk. Colab resets require redownload.

Please upload only the resulting `HAR_Stage12_Guide_Pseudobulk_Results.zip` into the conversation. A failed run will still generate diagnostic logs.


In [ ]:
!pip -q install 'gdown>=5,<7' 'h5py>=3' 'numpy>=1.24' 'pandas>=2' 'scipy>=1.10'
from pathlib import Path
import subprocess, json, os, sys, shutil, zipfile, traceback, time
ROOT=Path('/content/HAR_Stage12')
ROOT.mkdir(parents=True,exist_ok=True)
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
print('Python',sys.version,'free disk GB',round(shutil.disk_usage('/content').free/1e9,2))

In [ ]:
from pathlib import Path
Path('/content/HAR_Stage12/stream_h5ad_pseudobulk.py').write_text('"""GSE270828 h5ad disk-backed pseudobulk aggregator (no Seurat, no Drive mount).\n\nInput: publicly available, downloaded PRISM GSE270828 .h5ad. Only raw, integer\nRNA counts are accepted. H5AD X/layers/raw read directly through h5py.\nThis script does NOT perform differential expression or causal inference.\n"""\nfrom __future__ import annotations\nimport csv, gzip, json, os, sys, traceback\nfrom collections import Counter\nfrom pathlib import Path\nimport h5py\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\n\n\ndef decode_string_array(data):\n    return np.asarray([v.decode(\'utf-8\',errors=\'replace\') if isinstance(v,bytes) else str(v) for v in data], dtype=str)\n\n\ndef read_obs_column(g, name):\n    if name not in g:\n        raise KeyError(f"Missing required cell metadata field: {name}")\n    obj = g[name]\n    if isinstance(obj, h5py.Group) and \'codes\' in obj and \'categories\' in obj:\n        codes = obj[\'codes\'][:]\n        cats = decode_string_array(obj[\'categories\'][:])\n        return np.asarray([cats[int(c)] if 0 <= int(c) < len(cats) else \'\' for c in codes])\n    if isinstance(obj,h5py.Dataset):\n        return decode_string_array(obj[:])\n    raise ValueError(f\'Unsupported h5ad obs field: {name}\')\n\n\ndef read_genes(f, group_key, n_genes):\n    vg = f[\'raw/var\'] if group_key.startswith(\'raw/\') and \'raw/var\' in f else f[\'var\']\n    idxname = vg.attrs.get(\'_index\',\'_index\')\n    if isinstance(idxname,bytes): idxname=idxname.decode()\n    if idxname not in vg: idxname=\'_index\'\n    if idxname not in vg:\n        raise ValueError(f\'Gene IDs not found under {vg.name}\')\n    genes = decode_string_array(vg[idxname][:])\n    if len(genes) != n_genes: raise ValueError(\'Gene IDs / matrix dimensions mismatch\')\n    if len(set(genes)) != len(genes):\n        raise ValueError(\'Duplicated gene identifiers; manual resolution required before aggregation\')\n    return genes\n\n\ndef matrix_shape(x):\n    if isinstance(x,h5py.Dataset): return tuple(x.shape)\n    shape = x.attrs.get(\'shape\')\n    if shape is None and \'shape\' in x: shape=x[\'shape\'][:]\n    if shape is None: raise ValueError(f\'Missing sparse shape at {x.name}\')\n    return tuple(int(y) for y in shape)\n\n\ndef read_rows(x,start,end):\n    if isinstance(x,h5py.Dataset):\n        return np.asarray(x[start:end,:])\n    enc=x.attrs.get(\'encoding-type\',\'\')\n    if isinstance(enc,bytes): enc=enc.decode()\n    if enc != \'csr_matrix\':\n        raise ValueError(f\'{x.name}: {enc!r} encoding is not row-streamable with this code. Need CSR h5ad.\')\n    p=x[\'indptr\'][start:end+1]\n    off=int(p[0]); last=int(p[-1])\n    values=x[\'data\'][off:last]\n    cols=x[\'indices\'][off:last]\n    return sparse.csr_matrix((values,cols,p-off),shape=(end-start,matrix_shape(x)[1]))\n\n\ndef read_gene_columns_csc(x,start,end):\n    if isinstance(x,h5py.Dataset):return np.asarray(x[:,start:end])\n    enc=x.attrs.get(\'encoding-type\',\'\')\n    if isinstance(enc,bytes):enc=enc.decode()\n    if enc!=\'csc_matrix\':raise ValueError(\'CSC reader requires csc_matrix\')\n    p=x[\'indptr\'][start:end+1]\n    offset=int(p[0]);last=int(p[-1])\n    return sparse.csc_matrix((x[\'data\'][offset:last],x[\'indices\'][offset:last],p-offset),shape=(matrix_shape(x)[0],end-start))\n\n\ndef is_valid_count_block(mat):\n    arr = mat.data if sparse.issparse(mat) else np.asarray(mat)\n    return bool(np.isfinite(arr).all() and (arr>=0).all() and (np.abs(arr-np.rint(arr))<1e-6).all())\n\n\ndef choose_count_matrix(f,n_cells):\n    candidates=[\'layers/counts\',\'layers/raw_counts\',\'raw/X\',\'X\']\n    report=[]\n    for key in candidates:\n        if key not in f: continue\n        x=f[key]\n        try:\n            sh=matrix_shape(x)\n            if len(sh)!=2 or sh[0]!=n_cells: raise ValueError(f\'shape {sh} vs {n_cells} cells\')\n            if isinstance(x,h5py.Group):\n                enc=x.attrs.get(\'encoding-type\',\'\')\n                if isinstance(enc,bytes): enc=enc.decode()\n                if enc not in (\'csr_matrix\',\'csc_matrix\'): raise ValueError(f\'only CSR/CSC supported, found {enc}\')\n            # Sample multiple locations; every value is validated during aggregation.\n            ok=True\n            if isinstance(x,h5py.Group) and enc==\'csc_matrix\':\n                for st in sorted(set([0,max(0,sh[1]//3),max(0,2*sh[1]//3),max(0,sh[1]-16)])):\n                    if not is_valid_count_block(read_gene_columns_csc(x,st,min(st+16,sh[1]))):ok=False;break\n            else:\n                for st in sorted(set([0,max(0,n_cells//3),max(0,2*n_cells//3),max(0,n_cells-16)])):\n                    if not is_valid_count_block(read_rows(x,st,min(st+16,n_cells))):ok=False;break\n            report.append({\'key\':key,\'shape\':list(sh),\'looks_like_raw_counts\':ok})\n            if ok:return key,x,report\n        except Exception as exc:\n            report.append({\'key\':key,\'error\':str(exc)})\n    raise ValueError(\'No verified raw-count matrix found. Candidate inspection: \'+repr(report))\n\n\ndef aggregate_h5ad(h5ad_path,output_dir,chunk_size=256):\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    result={\'status\':\'started\',\'source\':str(h5ad_path),\'computes_p_values\':False,\'biological_replication_assumed\':False}\n    with h5py.File(h5ad_path,\'r\') as f:\n        if \'obs\' not in f:raise ValueError(\'Not a valid AnnData file: obs missing\')\n        obs=f[\'obs\']\n        rep=read_obs_column(obs,\'rep\') if \'rep\' in obs else read_obs_column(obs,\'replicate\')\n        har=read_obs_column(obs,\'har\')\n        numguide=read_obs_column(obs,\'num_guide\')\n        guides=read_obs_column(obs,\'guide\') if \'guide\' in obs else np.array([\'\']*len(har))\n        n_cells=len(har)\n        if n_cells<1000:raise ValueError(\'Expected full GSE270828 object; too few cells\')\n        key,x,source_report=choose_count_matrix(f,n_cells)\n        n_rows,n_genes=matrix_shape(x)\n        genes=read_genes(f,key,n_genes)\n        keep=(numguide==\'single_true\') & (rep!=\'\') & (har!=\'\')\n        if keep.sum()<1000:raise ValueError(\'Not enough single_true cells\')\n        label_all=np.char.add(np.char.add(rep.astype(str),\'__\'),har.astype(str))\n        groups=sorted(set(label_all[keep].tolist()))\n        gidx={k:i for i,k in enumerate(groups)}\n        membership=np.full(n_cells,-1,dtype=np.int32)\n        membership[keep]=np.array([gidx[x] for x in label_all[keep]],dtype=np.int32)\n        agg=np.zeros((len(groups),n_genes),dtype=np.int64)\n        count_by_group=np.zeros(len(groups),dtype=np.int64)\n        enc = x.attrs.get(\'encoding-type\',\'\') if isinstance(x,h5py.Group) else \'dense\'\n        if isinstance(enc,bytes):enc=enc.decode()\n        if enc==\'csc_matrix\':\n            # Column-streaming for CSC: no full matrix or densification of cells x genes.\n            selected_indices=np.where(keep)[0]\n            selected_groups=membership[selected_indices]\n            M_all=sparse.csr_matrix((np.ones(len(selected_groups),dtype=np.int64),(selected_groups,selected_indices)),shape=(len(groups),n_cells))\n            count_by_group=np.bincount(selected_groups,minlength=len(groups)).astype(np.int64)\n            gene_chunk=min(max(16,chunk_size//2),128)\n            for first in range(0,n_genes,gene_chunk):\n                last=min(first+gene_chunk,n_genes)\n                chunk=read_gene_columns_csc(x,first,last)\n                if not is_valid_count_block(chunk):\n                    raise ValueError(f\'Noninteger or invalid expression: {key}, gene columns {first}:{last}. Pseudobulk REFUSED.\')\n                block=(M_all @ chunk).toarray()\n                if not is_valid_count_block(block):raise ValueError(\'Invalid CSC aggregated counts\')\n                agg[:,first:last]=np.rint(block).astype(np.int64)\n                if first%(gene_chunk*50)==0:print(f\'Processed {last:,}/{n_genes:,} genes (CSC)\',flush=True)\n        else:\n            for start in range(0,n_cells,chunk_size):\n                stop=min(start+chunk_size,n_cells)\n                local=membership[start:stop]\n                mat=read_rows(x,start,stop)\n                if not is_valid_count_block(mat):\n                    raise ValueError(f\'Noninteger, nonfinite, or negative expression: {key}, rows {start}:{stop}. Pseudobulk REFUSED.\')\n                valid=local>=0\n                if not valid.any():continue\n                ids=local[valid]\n                count_by_group += np.bincount(ids,minlength=len(groups))\n                selected=mat[valid,:]\n                if not sparse.issparse(selected):selected=sparse.csr_matrix(selected)\n                M=sparse.csr_matrix((np.ones(len(ids),dtype=np.int64),(ids,np.arange(len(ids)))),shape=(len(groups),len(ids)))\n                block=(M @ selected).tocoo()\n                if not is_valid_count_block(block):raise ValueError(\'Aggregate count noninteger/invalid\')\n                # Update only observed nonzero entries; avoid per-chunk dense\n                # (groups x genes) materialization that can raise peak RAM.\n                np.add.at(agg,(block.row,block.col),np.rint(block.data).astype(np.int64))\n                if start%(chunk_size*50)==0:\n                    print(f\'Processed {stop:,}/{n_cells:,} cells; selected={count_by_group.sum():,}\',flush=True)\n        if int(count_by_group.sum())!=int(keep.sum()):raise AssertionError(\'Selected cell count mismatch\')\n        # Verify metadata nCount_RNA if present and represents raw RNA assay.\n        umi_check={}\n        if \'nCount_RNA\' in obs and isinstance(obs[\'nCount_RNA\'],h5py.Dataset):\n            try:\n                original=np.asarray(obs[\'nCount_RNA\'][:],dtype=float)\n                agg_total=int(agg.sum())\n                obs_total=float(original[keep].sum())\n                umi_check={\'sum_pseudobulk_raw_counts\':agg_total,\'sum_metadata_nCount_RNA\':obs_total,\'relative_difference\':float(abs(agg_total-obs_total)/max(obs_total,1))}\n            except Exception as exc:umi_check={\'error\':str(exc)}\n        meta=[]\n        for k,g in enumerate(groups):\n            re,ha=g.split(\'__\',1)\n            meta.append({\'group\':g,\'rep\':re,\'har\':ha,\'n_cells\':int(count_by_group[k]),\'control\':ha.casefold().startswith(\'non-target\') or ha.upper().startswith(\'NTC\')})\n        pd.DataFrame(meta).to_csv(output_dir/\'pseudobulk_group_metadata.csv\',index=False)\n        pd.DataFrame({\'gene_id\':genes}).to_csv(output_dir/\'gene_ids.csv\',index=False)\n        # Write the entire original-count aggregate; no top-gene truncation.\n        with gzip.open(output_dir/\'pseudobulk_RAW_COUNTS_all_genes.tsv.gz\',\'wt\',newline=\'\') as z:\n            writer=csv.writer(z,delimiter=\'\\t\')\n            writer.writerow([\'gene_id\',*groups])\n            for i,gene in enumerate(genes):writer.writerow([gene,*agg[:,i].tolist()])\n        result.update({\'status\':\'success\',\'n_cells_original\':int(n_cells),\'n_cells_selected\':int(keep.sum()),\'n_genes\':int(n_genes),\'n_groups\':int(len(groups)),\'n_har_labels\':len(set(har[keep].tolist())),\'n_replicate_labels\':len(set(rep[keep].tolist())),\'matrix_selected\':key,\'matrix_candidate_qc\':source_report,\'count_metadata_comparison\':umi_check,\'n_control_groups\':int(sum(row[\'control\'] for row in meta)),\'n_non_targeting_control_cells_selected\':int(sum(row[\'n_cells\'] for row in meta if row[\'control\']))})\n        return result\n\n\ndef synthetic_test(directory):\n    """Fixture-only validation. Never use synthetic fixture data as findings."""\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    src=directory/\'synthetic_fixture.h5ad\'\n    X=sparse.csr_matrix(np.asarray([[1,0,2,1],[4,0,1,0],[1,5,0,1],[0,2,0,0],[5,2,1,0],[2,3,4,0]]*200,dtype=np.int32))\n    reps=np.asarray([\'rep1\',\'rep1\',\'rep2\',\'rep2\',\'rep3\',\'rep3\']*200)\n    hars=np.asarray([\'HAR_A\',\'Non-Targeting\',\'HAR_A\',\'Non-Targeting\',\'HAR_A\',\'Non-Targeting\']*200)\n    true=np.asarray([\'single_true\']*len(hars))\n    def h5strings(g,key,values):g.create_dataset(key,data=np.asarray(values,dtype=h5py.string_dtype(\'utf-8\')))\n    with h5py.File(src,\'w\') as f:\n        obs=f.create_group(\'obs\');h5strings(obs,\'rep\',reps);h5strings(obs,\'har\',hars);h5strings(obs,\'num_guide\',true)\n        var=f.create_group(\'var\');h5strings(var,\'_index\',[\'g1\',\'g2\',\'g3\',\'g4\'])\n        x=f.create_group(\'X\');x.attrs[\'encoding-type\']=\'csr_matrix\';x.attrs[\'shape\']=(X.shape[0],X.shape[1])\n        x.create_dataset(\'data\',data=X.data);x.create_dataset(\'indices\',data=X.indices);x.create_dataset(\'indptr\',data=X.indptr)\n    r=aggregate_h5ad(src,directory/\'outputs\',chunk_size=17)\n    m=pd.read_csv(directory/\'outputs/pseudobulk_group_metadata.csv\')\n    assert r[\'n_cells_selected\']==1200 and r[\'n_groups\']==6 and int(m[\'n_cells\'].sum())==1200\n    with gzip.open(directory/\'outputs/pseudobulk_RAW_COUNTS_all_genes.tsv.gz\',\'rt\') as z:\n        rr=pd.read_csv(z,sep=\'\\t\')\n    assert len(rr)==4 and int(rr.drop(columns=[\'gene_id\']).to_numpy().sum())==int(X.sum())\n    return {\'synthetic_smoke_test\':\'PASS\',\'n_cells\':1200,\'n_genes\':4,\'n_groups\':6}\n\n\nif __name__==\'__main__\':\n    if len(sys.argv)>1 and sys.argv[1]==\'--smoke\':\n        print(json.dumps(synthetic_test(sys.argv[2] if len(sys.argv)>2 else \'/tmp/stage10c_test\'),indent=2))\n    else:\n        if len(sys.argv)<3:raise SystemExit(\'usage: python stream_h5ad_pseudobulk.py DATA.h5ad OUTDIR\')\n        try:\n            report=aggregate_h5ad(sys.argv[1],sys.argv[2]);print(json.dumps(report,indent=2))\n            Path(sys.argv[2],\'QC_and_STATUS.json\').write_text(json.dumps(report,indent=2))\n        except Exception as e:\n            out=Path(sys.argv[2]);out.mkdir(parents=True,exist_ok=True)\n            status={\'status\':\'failed\',\'error\':str(e),\'traceback\':traceback.format_exc(),\'fabricated_data\':False}\n            (out/\'QC_and_STATUS.json\').write_text(json.dumps(status,indent=2))\n            raise\n',encoding='utf-8')
Path('/content/HAR_Stage12/Stage12_Guide_Pseudobulk_Extractor.py').write_text('"""Stage 12 guide × replicate pseudobulk extractor for verified GSE270828 h5ad.\nIndependent guide concordance requires this (Stage 10D HAR-level aggregation collapses guides).\nAll counts are raw integer, selected single_true cells only. No differential expression.\n"""\nfrom pathlib import Path\nimport csv,gzip,json,sys,traceback\nimport numpy as np,pandas as pd,h5py\nfrom scipy import sparse\nfrom stream_h5ad_pseudobulk import (read_obs_column,choose_count_matrix,matrix_shape,read_genes,read_rows,read_gene_columns_csc,is_valid_count_block)\n\ndef aggregate(path,outdir,chunk_size=256):\n outdir=Path(outdir);outdir.mkdir(exist_ok=True,parents=True)\n with h5py.File(path,\'r\') as f:\n  obs=f[\'obs\']\n  rep=read_obs_column(obs,\'rep\') if \'rep\' in obs else read_obs_column(obs,\'replicate\')\n  har=read_obs_column(obs,\'har\')\n  guide=read_obs_column(obs,\'guide\')\n  single=read_obs_column(obs,\'num_guide\')\n  n_cells=len(rep)\n  key,x,report=choose_count_matrix(f,n_cells)\n  n_rows,n_genes=matrix_shape(x)\n  genes=read_genes(f,key,n_genes)\n  keep=(single==\'single_true\') & (rep!=\'\') & (har!=\'\') & (guide!=\'\')\n  if not keep.any():raise ValueError(\'No eligible single_true cells with guide labels\')\n  selected=pd.DataFrame({\'rep\':rep[keep],\'har\':har[keep],\'guide\':guide[keep]})\n  guide_map=selected[[\'guide\',\'har\']].drop_duplicates()\n  ambiguous=guide_map.groupby(\'guide\').har.nunique()\n  if (ambiguous>1).any():\n    guide_map.to_csv(outdir/\'ambiguous_guide_to_har_mappings.csv\',index=False)\n    raise ValueError(\'One or more guides map to multiple HAR labels; manual reconciliation required\')\n  groupmeta=selected.groupby([\'rep\',\'guide\',\'har\'],as_index=False).size().rename(columns={\'size\':\'n_cells\'}).sort_values([\'rep\',\'guide\'])\n  groups=[(r.rep,r.guide) for _,r in groupmeta.iterrows()]\n  groupmap={k:i for i,k in enumerate(groups)}\n  assert len(groupmap)==len(groups)\n  labels=np.full(n_cells,-1,dtype=np.int32)\n  labels[np.where(keep)[0]]=np.array([groupmap[(r,g)] for r,g in zip(rep[keep],guide[keep])],dtype=np.int32)\n  n_groups=len(groups)\n  # Small dense final aggregate only: ~1100 × ~20k integer entries (175MB).\n  agg=np.zeros((n_groups,n_genes),dtype=np.int64)\n  group_counts=np.zeros(n_groups,dtype=np.int64)\n  enc=x.attrs.get(\'encoding-type\',\'dense\') if isinstance(x,h5py.Group) else \'dense\'\n  if isinstance(enc,bytes):enc=enc.decode()\n  if enc==\'csc_matrix\':\n    selected_indices=np.where(keep)[0]\n    group_ids=labels[selected_indices]\n    membership=sparse.csr_matrix((np.ones(len(group_ids),dtype=np.int64),(group_ids,selected_indices)),shape=(n_groups,n_cells))\n    group_counts=np.bincount(group_ids,minlength=n_groups)\n    for first in range(0,n_genes,128):\n      last=min(first+128,n_genes)\n      chunk=read_gene_columns_csc(x,first,last)\n      if not is_valid_count_block(chunk):raise ValueError(\'Invalid gene count data\')\n      block=(membership @ chunk).toarray()\n      if not is_valid_count_block(block):raise ValueError(\'Invalid guide aggregate\')\n      agg[:,first:last]=np.rint(block).astype(np.int64)\n      if first%1280==0:print(f\'GUIDE_AGGREGATION: {last}/{n_genes} genes\',flush=True)\n  else:\n    for start in range(0,n_cells,chunk_size):\n      end=min(start+chunk_size,n_cells)\n      grp=labels[start:end]\n      sub=read_rows(x,start,end)\n      if not is_valid_count_block(sub):raise ValueError(\'Noninteger or invalid count input\')\n      valid=grp>=0\n      if valid.any():\n        gids=grp[valid];group_counts+=np.bincount(gids,minlength=n_groups)\n        sub=sub[valid,:]\n        if not sparse.issparse(sub):sub=sparse.csr_matrix(sub)\n        membership=sparse.csr_matrix((np.ones(len(gids),dtype=np.int64),(gids,np.arange(len(gids)))),shape=(n_groups,len(gids)))\n        block=(membership @ sub).tocoo()\n        if not is_valid_count_block(block):raise ValueError(\'Invalid group count data\')\n        np.add.at(agg,(block.row,block.col),np.rint(block.data).astype(np.int64))\n      if start%(chunk_size*100)==0:print(f\'GUIDE_AGGREGATION: {end}/{n_cells} cells\',flush=True)\n  assert int(group_counts.sum())==int(keep.sum())\n  assert np.array_equal(groupmeta.n_cells.to_numpy(),group_counts)\n  raw_sum=int(agg.sum())\n  control=groupmeta.har.str.contains(\'Non-Targeting\',case=False,regex=False)\n  groupmeta.insert(0,\'group\',[r+\'__\'+g for r,g in groups])\n  groupmeta[\'control\']=control\n  groupmeta.to_csv(outdir/\'guide_pseudobulk_group_metadata.csv\',index=False)\n  pd.DataFrame({\'gene_id\':genes}).to_csv(outdir/\'gene_ids.csv\',index=False)\n  with gzip.open(outdir/\'guide_pseudobulk_RAW_COUNTS.tsv.gz\',\'wt\',newline=\'\') as fz:\n    wr=csv.writer(fz,delimiter=\'\\t\');wr.writerow([\'gene_id\',*groupmeta.group.tolist()])\n    for i,gn in enumerate(genes):wr.writerow([gn,*agg[:,i]])\n  pd.DataFrame({\'guide\':guide_map.guide,\'har\':guide_map.har}).to_csv(outdir/\'guide_to_har_mapping.csv\',index=False)\n  num_ntc=int(groupmeta.loc[groupmeta.control,\'n_cells\'].sum())\n  out={\'status\':\'success\',\'n_cells_original\':n_cells,\'n_cells_selected\':int(keep.sum()),\'n_genes\':int(n_genes),\n       \'n_guide_rep_groups\':int(n_groups),\'n_distinct_guides\':int(guide_map.guide.nunique()),\n       \'n_distinct_har_labels\':int(groupmeta.har.nunique()),\'n_NTC_cells\':num_ntc,\'raw_count_sum\':raw_sum,\n       \'matrix_selected\':key,\'source_matrix_validation\':report,\'compute_p_values\':False,\n       \'replicate_independence_verified\':False,\'purpose\':\'guide-level concordance QC only\'}\n  if \'nCount_RNA\' in obs:\n    y=obs[\'nCount_RNA\'][:]\n    out[\'raw_UMIs_equal_cell_metadata\']=bool(abs(float(y[keep].sum())-raw_sum)<.1)\n    if not out[\'raw_UMIs_equal_cell_metadata\']:raise ValueError(\'Aggregate counts mismatch input cell metadata\')\n  (outdir/\'S0_Guide_QC.json\').write_text(json.dumps(out,indent=2))\n  return out\n\nif __name__==\'__main__\':\n if len(sys.argv)<3:raise SystemExit(\'Usage: Stage12_Guide_Pseudobulk_Extractor.py SOURCE.h5ad OUTPUT_DIR\')\n dest=Path(sys.argv[2]);dest.mkdir(parents=True,exist_ok=True)\n try:\n   print(json.dumps(aggregate(sys.argv[1],dest),indent=2))\n except Exception as e:\n   (dest/\'S0_Guide_QC.json\').write_text(json.dumps({\'status\':\'failed\',\'error\':str(e),\'traceback\':traceback.format_exc(),\'no_invented_data\':True},indent=2))\n   raise\n',encoding='utf-8')
print('Both self-contained Python scripts saved')

In [ ]:
import gdown,h5py,json,traceback
SOURCE_ID='1BwQK4rNA6Ouy1CyhzNosoCDyWKov35Ea'
CANDIDATES=[Path('/content/HAR_Stage10D/GSE270828_source.h5ad'),Path('/content/HAR_Stage10C/GSE270828_source.h5ad'),Path('/content/HAR_Stage12/GSE270828_source.h5ad')]
file=None
source_status={'source_id':SOURCE_ID,'personal_Google_Drive_mount':False,'status':'started'}
try:
    for c in CANDIDATES:
        if c.is_file() and c.stat().st_size>3_000_000_000:
            try:
                with h5py.File(c,'r') as h: assert 'obs' in h and 'X' in h
                file=c;break
            except Exception as e: print('Invalid candidate file',c,repr(e))
    if file is None:
        dest=CANDIDATES[-1]
        print('Downloading public PRISM GSE270828 h5ad directly; no personal Drive mount:',dest)
        got=gdown.download(id=SOURCE_ID,output=str(dest),quiet=False)
        if not got or not dest.exists() or dest.stat().st_size<3_000_000_000:
            raise RuntimeError(f'Public source download incomplete: {got}')
        with h5py.File(dest,'r') as h: assert 'obs' in h and 'X' in h
        file=dest
    source_status.update(status='ready',file=str(file),bytes=file.stat().st_size,verified_h5ad=True)
    print('SOURCE_READY:',file,file.stat().st_size,'bytes')
except Exception as e:
    source_status.update(status='failed',error=str(e),traceback=traceback.format_exc())
    print('SOURCE_DOWNLOAD_FAILURE:',str(e))
(ROOT/'SOURCE_CHECK.json').write_text(json.dumps(source_status,indent=2))

In [ ]:
if file is None:
    print('GUIDE_AGGREGATION_SKIPPED: public source unavailable; diagnostic export will follow.')
    (ROOT/'stage12_run.log').write_text('Guide aggregation skipped. See SOURCE_CHECK.json')
else:
    cmd=[sys.executable,str(ROOT/'Stage12_Guide_Pseudobulk_Extractor.py'),str(file),str(OUT)]
    with open(ROOT/'stage12_run.log','w') as log:
        proc=subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT,text=True)
    print('PROCESS_EXIT_CODE:',proc.returncode)
    print((ROOT/'stage12_run.log').read_text()[-6000:])
    print('Output files:',[(p.name,p.stat().st_size) for p in OUT.iterdir()])

In [ ]:
# Always export diagnostic output whether the analysis succeeds or fails.
import zipfile,json
archive=Path('/content/HAR_Stage12_Guide_Pseudobulk_Results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.name)
    for extra in [ROOT/'stage12_run.log',ROOT/'SOURCE_CHECK.json',ROOT/'Stage12_Guide_Pseudobulk_Extractor.py',ROOT/'stream_h5ad_pseudobulk.py']:
        if extra.exists():z.write(extra,extra.name)
print('EXPORT_READY:',archive,round(archive.stat().st_size/1e6,2),'MB')
from google.colab import files
files.download(str(archive))